In [ ]:
## Імпорт бібліотек

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
## Завантаження датасету

In [2]:
melbourne_file_path = 'melb_data.csv'
melbourne_data = pd.read_csv(melbourne_file_path)
melbourne_data.describe()

,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,13580.000000,1.358000e+04,13580.000000,13580.000000,13580.000000,13580.000000,13518.000000,13580.000000,7130.000000,8205.000000,13580.000000,13580.000000,13580.000000
mean,2.937997,1.075684e+06,10.137776,3105.301915,2.914728,1.534242,1.610075,558.416127,151.967650,1964.684217,-37.809203,144.995216,7454.417378
std,0.955748,6.393107e+05,5.868725,90.676964,0.965921,0.691712,0.962634,3990.669241,541.014538,37.273762,0.079260,0.103916,4378.581772
min,1.000000,8.500000e+04,0.000000,3000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1196.000000,-38.182550,144.431810,249.000000
25%,2.000000,6.500000e+05,6.100000,3044.000000,2.000000,1.000000,1.000000,177.000000,93.000000,1940.000000,-37.856822,144.929600,4380.000000
50%,3.000000,9.030000e+05,9.200000,3084.000000,3.000000,1.000000,2.000000,440.000000,126.000000,1970.000000,-37.802355,145.000100,6555.000000
75%,3.000000,1.330000e+06,13.000000,3148.000000,3.000000,2.000000,2.000000,651.000000,174.000000,1999.000000,-37.756400,145.058305,10331.000000
max,10.000000,9.000000e+06,48.100000,3977.000000,20.000000,8.000000,10.000000,433014.000000,44515.000000,2018.000000,-37.408530,145.526350,21650.000000


In [ ]:
## Визначення прогнозованої змінної - ціни житла

In [3]:
y = melbourne_data.Price
y.head()

0    1480000.0
1    1035000.0
2    1465000.0
3     850000.0
4    1600000.0
Name: Price, dtype: float64

In [ ]:
## Вибір характеристики, які будемо використовувати для прогнозування цін

In [4]:
melbourne_features = ['Rooms','Bathroom','Landsize','BuildingArea','YearBuilt']
X = melbourne_data[melbourne_features]
X.head()

,Rooms,Bathroom,Landsize,BuildingArea,YearBuilt
0,2,1.0,202.0,NaN,NaN
1,2,1.0,156.0,79.0,1900.0
2,3,2.0,134.0,150.0,1900.0
3,3,2.0,94.0,NaN,NaN
4,4,1.0,120.0,142.0,2014.0


In [ ]:
## Розділення даних на тренувальні та валідаційні для правильної оцінки моделі

In [5]:
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=0)

In [ ]:
## Модель Decision Tree

In [6]:
melbourne_model = DecisionTreeRegressor(random_state=1)
melbourne_model.fit(train_X, train_y)
val_predictions = melbourne_model.predict(val_X)
print("Decision Tree MAE:", mean_absolute_error(val_y, val_predictions))

Decision Tree MAE: 409286.9187577298


In [ ]:
## Підбір розміоу дерева

In [7]:
def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=0)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    mae = mean_absolute_error(val_y, preds_val)
    return mae

for max_leaf_nodes in [5, 25, 50, 100, 250, 500]:
    my_mae = get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y)
    print("Max leaf nodes:", max_leaf_nodes, "  MAE:", my_mae)

Max leaf nodes: 5   MAE: 397665.11983400927
Max leaf nodes: 25   MAE: 396983.6356930905
Max leaf nodes: 50   MAE: 397516.4664198829
Max leaf nodes: 100   MAE: 397516.4664198829
Max leaf nodes: 250   MAE: 397516.4664198829
Max leaf nodes: 500   MAE: 397516.4664198829


In [ ]:
## Проноз для 5 будинків

In [8]:
print("Прогноз для перших 5 будинків:")
print(X.head())
print("Прогнозовані ціни:")
print(melbourne_model.predict(X.head()))

Прогноз для перших 5 будинків:
   Rooms  Bathroom  Landsize  BuildingArea  YearBuilt
0      2       1.0     202.0           NaN        NaN
1      2       1.0     156.0          79.0     1900.0
2      3       2.0     134.0         150.0     1900.0
3      3       2.0      94.0           NaN        NaN
4      4       1.0     120.0         142.0     2014.0
Прогнозовані ціни:
[ 923704.85215794  923704.85215794  923704.85215794  923704.85215794
 1411639.54434538]


In [ ]:
## При першому запуску створені моделі Random Forest було виявлено ошибку, тому було виконано перевірку пропусків

In [10]:
print(X.isnull().sum()) 
print(y.isnull().sum()) # Перевірка пропусків

Rooms              0
Bathroom           0
Landsize           0
BuildingArea    6450
YearBuilt       5375
dtype: int64
0


In [11]:
X = X.dropna() # Видалення рядків, де пропущені значення
y = y[X.index] 

In [ ]:
## Повторна спроба створення моделі Random Forest

In [12]:
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=0)
forest_model = RandomForestRegressor(random_state=1)
forest_model.fit(train_X, train_y)
melb_preds = forest_model.predict(val_X)
print("Random Forest MAE:", mean_absolute_error(val_y, melb_preds))

Random Forest MAE: 285516.8253913958
